## Controlled-experiment Using Pre-Experiment Data (CUPED)

### Idea

- In A/B testing, we frequently encounter situations where we are unable to make a statistically significant conclusion at some desired confidence level (i.e. 5%)

- This happens because the treatment effect has large variance, so the standard error on the estimate $\delta = \bar{x}_{\text{Treatment}} - \bar{x}_{\text{Control}}$ is also large

- CUPED provides a way for us to reduce the variance of $\delta$ by reducing its standard error, thereby allowing us to reach statistical significance more readily

### How does CUPED work?

- Let's assume we want to run an experiment on the effect redesigning the product page on your e-commerce site

- To do this, we perform an A/B test. The control group sees the old product page, and the treatment group sees the new one. We want to compare the average spend during the experimental week for the control vs the treatment groups. However, because you are small company, you don't have too many observations. But you still want to reduce variance to make some inference.

- This is the perfect set up for CUPED

- NOTE: The hypothesis being tested here is to estimate the difference between treatment and control groups due to the **intervention** in the post period, NOT the difference in spend between the pre and post period 
    - i.e. We are interested in the difference between treatment and control in time period $t_2$, not the effect of intervention as measured by spend in $t_2$ - spend in $t_1$

- Let's assume the following dataset. 

- We have a set of users, their spending before the intervention period, and their spending during the intervention period. Within the intervention period, they are split into control and treatment

In [ ]:
import numpy as np
import pandas as pd

sample_data = pd.DataFrame({
    'user': ['u1','u2','u3','u4','u5','u6'],
    'pre_spend': [0, 20, 40, 60, 80, 100],
    'post_spend': [0, 25, 60, 90, 100, 135],
    'treatment': [0, 0, 1, 1, 0, 1]
})

- Now, put yourself in the position of someone running a vanilla A/B test. If we want to find out if our treatment has successfully increased spending, we can simply default to the t-test, which measures the difference of the means between treatment and control groups 

In [19]:
from scipy import stats
control = sample_data.query('treatment == 0')['post_spend']
treatment = sample_data.query('treatment == 1')['post_spend']

stat, p = stats.ttest_ind(treatment, control, equal_var=False)
effect = treatment.mean() - control.mean()

print(f'{effect=}, {stat=}, {p=}')

effect=np.float64(53.333333333333336), stat=np.float64(1.4368424162141993), p=np.float64(0.2306550482896477)


- But, shock and horror, your p-value is 0.14, which indicates that there is no statistically significant positive effect!
    
- Let's look at the individual observations to understand why this is happening:
    - Control:
        - U1: 0 --> 0
        - U2: 20 --> 25
        - U5: 80 --> 100
    - Treatment:
        - U3: 40 --> 60
        - U4: 60 --> 90
        - U6: 100 --> 135

- Treatment seems to have increased the spending for all 3 users. But we have 1 particular user in the control group who increased spending from 20 --> 90 despite receiving no intervention. This means that the post-spend group has significance variance from person to person, which drowns out the treatment effect!

- That is, the set of post values `[0, 25, 100, 60, 90, 135]` has large variance, which drowns out any effect due to treatment

- Since the t-test works on post-treatment outcome variance, and since the variability within groups is large relative to the mean difference between groups, the standard error becomes large. The large standard error leads to a small t-statistic, which leads to a large p value

- But if you study the data a little more carefully, you find that the though the variance within the post group is high, a lot of this variance is actually predictable! Looking within the same user, if the user's spend in the pre period is high, the user spend in the post period also tends to be high!
    - Control:
        - U1: 0 --> 0
        - U2: 20 --> 25
        - U5: 80 --> 100
    - Treatment:
        - U3: 40 --> 60
        - U4: 60 --> 90
        - U6: 100 --> 135

- How does this help us? As it turns out, since a lot of the variance in the post-period can be explained by the pre-period, why not "remove" that component of variance from the post period? By doing this, we can better "isolate" the variation that results from the treatment! 

- Let's do this with the most basic idea: OLS
    - We estimate an OLS predictor of postspend ~ prespend
    - Based on this estimate, CUPED subtracts the component predicted by pre-spend
    - Finally, we do the same t-test, but using the adjusted post-spend data

- The idea here is that CUPED removes predictable individual-level variation in post-spend using the pre-spend signal

In [16]:
### Estimate OLS coefficient of post_spend ~ pre_spend
theta = np.cov(sample_data['post_spend'], sample_data['pre_spend'], bias=True)[0,1] / np.var(sample_data['pre_spend']) 

### Use theta to "debias" the post spend
sample_data['post_spend_cuped'] = (
    sample_data['post_spend'] - 
    (theta * (sample_data['pre_spend'] - sample_data['pre_spend'].mean()))
)
sample_data

,user,pre_spend,post_spend,treatment,post_spend_cuped
0,u1,0,0,0,66.428571
1,u2,20,25,0,64.857143
2,u3,40,60,1,73.285714
3,u4,60,90,1,76.714286
4,u5,80,100,0,60.142857
5,u6,100,135,1,68.571429


In [ ]:
from scipy import stats
control = sample_data.query('treatment == 0')['post_spend']
treatment = sample_data.query('treatment == 1')['post_spend']
control_cuped = sample_data.query('treatment == 0')['post_spend_cuped']
treatment_cuped = sample_data.query('treatment == 1')['post_spend_cuped']

stat, p = stats.ttest_ind(treatment, control, equal_var=False)
effect = treatment.mean() - control.mean()

stat_cuped, p_cuped = stats.ttest_ind(treatment_cuped, control_cuped, equal_var=False)
effect_cuped = treatment_cuped.mean() - control_cuped.mean()


print(f'{effect=}, {stat=}, {p=}')
print(f'{effect_cuped=}, {stat_cuped=}, {p_cuped=}')

effect=np.float64(53.333333333333336), stat=np.float64(1.4368424162141993), p=np.float64(0.2306550482896477)
effect_cuped=np.float64(9.047619047619044), stat_cuped=np.float64(2.992961138600232), p_cuped=np.float64(0.04269895650146775)


- MAGIC! Now with CUPED, our p value is statistically significant!

### Limits of CUPED

- For CUPED to work properly, some conditions must be met

- Let's think through our workflow in the section above: 
    - We collect data in 2 time periods, before and after an intervention event, for both the treatment and control groups
    - We claim that the metric pre-treatment predicts the metric in the metric post-treatment in the absence of intervention. So if the pre-treatment value is low, the post treatment value should be low in the absence of intervention
    - Because of this, we can adjust the effect of the post treatment value by its predictions from pre-treatment information, which would hopefully reduce the variance of our data
    

- `metric pre-treatment correlates with the metric post-treatment in the absence of intervention`
    - This is an assumption you'll need to justify, probably using some domain specific reasons to think so
    - For example, in a setting of buying a TV, someone buying a TV in the pre period will most likely NOT be buying a TV in the next period. Thus, (negative) correlation!

- Furthermore, since we use the same OLS value to adjust our post-period outcome, we have to assume that the intervention does not affect the OLS coefficient estimated! That is, the treatment doesn't affect the covariate